# 03.02 — Benchmark Qwen3-4B using SageMaker AI Benchmarking Service

This notebook uses the **SageMaker AI Benchmarking Service** to run standardised AIPerf benchmarks.

## Choose your path

| Path | Run if you completed | Jump to |
|---|---|---|
| **Part A — IC-only** | Task 02.02 (Multi-LoRA with Inference Components) | [Part A](#part-a) |
| **Part B — Non-IC only** | Task 02.01 (Standard fine-tuned endpoint) | [Part B](#part-b) |
| **Full comparison** | Both 02.01 and 02.02 | Run both parts |

Both parts share the same **Setup** (Steps 1–4) and **Results** (Steps 7–9) sections. Run Setup first, then jump to whichever part applies to you.

The service produces:
- Time to First Token (TTFT) at p50/p90/p99
- Request latency end-to-end
- Output token throughput (tokens/s)
- Inter-token latency (ITL)

> **Tip:** Endpoint names are loaded automatically from `%store` if you ran 02.01/02.02. You can also set them manually in Step 2.

## Step 1 — Install dependencies

Upgrade boto3 to ensure the latest Benchmarking API is available.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3 sagemaker matplotlib pandas

## Step 2 — Imports and configuration

Endpoint names are loaded automatically from `%store` if you ran 02.01/02.02.
You can also set them manually by uncommenting the override lines below.

> **Only fill in the endpoint(s) for the path you are taking.** Leave the others as `None`.

In [ ]:
import json, time, io, tarfile, uuid, boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, Markdown, Image, clear_output

session    = boto3.Session()
region     = session.region_name
sm         = boto3.client("sagemaker")
sm_runtime = boto3.client("sagemaker-runtime")
s3_client  = boto3.client("s3")

print(f"Region: {region}")

In [ ]:
# ── Restore endpoint names from 02.01 / 02.02 ────────────────────────────────
%store -r TUNED_ENDPOINT_NAME
%store -r IC_ENDPOINT_NAME
%store -r BASE_IC_NAME
%store -r ADAPTER_IC_NAME
%store -r FINQA_IC_NAME

# %store -r prints a warning but does NOT raise when a variable is missing —
# it simply leaves the name undefined. Default any undefined names to None.
TUNED_ENDPOINT_NAME = locals().get("TUNED_ENDPOINT_NAME", None)
IC_ENDPOINT_NAME    = locals().get("IC_ENDPOINT_NAME",    None)
BASE_IC_NAME        = locals().get("BASE_IC_NAME",        None)
ADAPTER_IC_NAME     = locals().get("ADAPTER_IC_NAME",     None)
FINQA_IC_NAME       = locals().get("FINQA_IC_NAME",       None)

# ── Manual overrides — uncomment and fill in if %store is not available ───────
# TUNED_ENDPOINT_NAME = "Qwen3-4B-Instruct-2507-sft-XXXXXXXX"  # from 02.01
# IC_ENDPOINT_NAME    = "qwen3-4b-YYMMDD-HHMMSS"               # from 02.02
# BASE_IC_NAME        = "base-qwen3-4b-YYMMDD-HHMMSS"          # from 02.02
# ADAPTER_IC_NAME     = "adapter-qwen3-4b-YYMMDD-HHMMSS"       # from 02.02
# FINQA_IC_NAME       = "finqa-adapter-qwen3-4b-YYMMDD-HHMMSS" # from 02.02

NON_IC_ENDPOINT = TUNED_ENDPOINT_NAME
IC_ENDPOINT     = IC_ENDPOINT_NAME

# ── Summary ───────────────────────────────────────────────────────────────────
print("Endpoint configuration:")
print(f"  Non-IC endpoint : {NON_IC_ENDPOINT or '(not set)'}")
print(f"  IC endpoint     : {IC_ENDPOINT     or '(not set)'}")
if IC_ENDPOINT:
    print(f"    BASE_IC_NAME    : {BASE_IC_NAME}")
    print(f"    ADAPTER_IC_NAME : {ADAPTER_IC_NAME}")
    print(f"    FINQA_IC_NAME   : {FINQA_IC_NAME}")

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess      = Session()
S3_OUTPUT = f"s3://{sess.default_bucket()}/benchmark-results/ic-vs-non-ic/"
ROLE_ARN  = get_execution_role(sess, use_default=True)
RUN_ID    = datetime.now().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]

print(f"S3 output : {S3_OUTPUT}")
print(f"Role ARN  : {ROLE_ARN}")
print(f"Run ID    : {RUN_ID}")

## Step 3 — Verify IAM trust policy

The execution role must allow `sagemaker.amazonaws.com` to assume it so the Benchmarking Service can run AIPerf as a Training Job in your account. This cell checks and patches the trust policy if needed.

In [ ]:
iam       = boto3.client("iam")
role_name = ROLE_ARN.split("/")[-1]

trust = iam.get_role(RoleName=role_name)["Role"]["AssumeRolePolicyDocument"]
principals = [
    s.get("Principal", {}).get("Service", "")
    for s in trust.get("Statement", [])
]
flat = [p for item in principals for p in (item if isinstance(item, list) else [item])]

if "sagemaker.amazonaws.com" in flat:
    print("Trust policy already allows sagemaker.amazonaws.com")
else:
    trust["Statement"].append({
        "Effect": "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action": "sts:AssumeRole",
    })
    iam.update_assume_role_policy(RoleName=role_name, PolicyDocument=json.dumps(trust))
    print(f"Added sagemaker.amazonaws.com to trust policy for: {role_name}")

## Step 4 — Create shared workload config

A single workload config is reused across all benchmark jobs so results are directly comparable.

**Parameters:**
| Parameter | Workshop value | Production value | Notes |
|---|---|---|---|
| `public_dataset` | `sharegpt` | `sharegpt` | Realistic chat prompts |
| `concurrency` | `3` | `10` | Simultaneous requests |
| `request_count` | `50` | `300` | Total requests per job |
| `output_tokens_mean` | `100` | `200` | Average output length |
| `extra_inputs` | `ignore_eos:true` | `ignore_eos:true` | Forces full output length |

> **Workshop mode:** Values are tuned for ~5 min per job instead of ~20 min. Results are still statistically meaningful for comparing IC vs Non-IC — just with a smaller sample. Increase `request_count` to 300 and `concurrency` to 10 for production-grade benchmarks.

After running this cell, jump to the section that matches your path:
- **[Part A — IC Benchmark (02.02)](#part-a)** — if you completed task 02.02
- **[Part B — Non-IC Benchmark (02.01)](#part-b)** — if you completed task 02.01

In [ ]:
WORKLOAD_NAME = f"wl-ic-bench-{RUN_ID}"

workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "public_dataset":            "sharegpt",
        "output_tokens_mean":        100,   # 200 for production
        "output_tokens_stddev":      10,
        "extra_inputs":              "ignore_eos:true",
        "concurrency":               3,     # 10 for production
        "request_count":             50,    # 300 for production
    },
}

wl_resp = sm.create_ai_workload_config(
    AIWorkloadConfigName=WORKLOAD_NAME,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Workload config: {wl_resp['AIWorkloadConfigArn']}")

## Step 5 — Submit benchmark jobs

One benchmark job is submitted per available target. Targets with `None` endpoints are automatically skipped — no error.

| Target | Required | Endpoint | IC |
|---|---|---|---|
| Non-IC (merged SFT) | 02.01 | `NON_IC_ENDPOINT` | — |
| IC — Base model | 02.02 | `IC_ENDPOINT` | `BASE_IC_NAME` |
| IC — Medical adapter | 02.02 | `IC_ENDPOINT` | `ADAPTER_IC_NAME` |
| IC — FinQA adapter | 02.02 | `IC_ENDPOINT` | `FINQA_IC_NAME` |

The `InferenceComponents` field in `BenchmarkTarget` is optional — it is omitted for non-IC endpoints.

In [ ]:
# ── Shared helper — used by both Part A and Part B ────────────────────────────
def submit_benchmark(label, endpoint_name, ic_name=None):
    job_name = f"bmk-{label.lower().replace(' ', '-').replace('/', '-')[:20]}-{RUN_ID}"
    target = {"Endpoint": {"Identifier": endpoint_name}}
    if ic_name:
        target["Endpoint"]["InferenceComponents"] = [{"Identifier": ic_name}]

    resp = sm.create_ai_benchmark_job(
        AIBenchmarkJobName=job_name,
        BenchmarkTarget=target,
        OutputConfig={"S3OutputLocation": S3_OUTPUT},
        AIWorkloadConfigIdentifier=WORKLOAD_NAME,
        RoleArn=ROLE_ARN,
    )
    print(f"  Submitted '{label}': {resp['AIBenchmarkJobArn']}")
    return {"label": label, "job_name": job_name, "endpoint": endpoint_name, "ic": ic_name}

# jobs list accumulates submissions from Part A and/or Part B
jobs = []

---
<a id='part-a'></a>
## Part A — IC Benchmark (Task 02.02)

> **Run this section if you completed task 02.02** (Multi-LoRA with Inference Components).
> Skip to [Part B](#part-b) if you only completed 02.01.

Submits one benchmark job per IC target:
- Base Qwen3-4B model IC
- Medical reasoning LoRA adapter IC
- FinQA financial reasoning LoRA adapter IC

All three share the same endpoint and the same workload config created in Step 4.

In [ ]:
# ── Part A: Submit IC benchmark jobs ─────────────────────────────────────────
if not IC_ENDPOINT:
    print("⚠️  IC_ENDPOINT is not set. Did you run task 02.02?")
    print("   Set IC_ENDPOINT_NAME manually in Step 2 and re-run.")
else:
    ic_targets = []

    if BASE_IC_NAME:
        ic_targets.append(("IC - Base model",      IC_ENDPOINT, BASE_IC_NAME))
    else:
        print("Skipping IC - Base model: BASE_IC_NAME not set.")

    if ADAPTER_IC_NAME:
        ic_targets.append(("IC - Medical adapter", IC_ENDPOINT, ADAPTER_IC_NAME))
    else:
        print("Skipping IC - Medical adapter: ADAPTER_IC_NAME not set.")

    if FINQA_IC_NAME:
        ic_targets.append(("IC - FinQA adapter",   IC_ENDPOINT, FINQA_IC_NAME))
    else:
        print("Skipping IC - FinQA adapter: FINQA_IC_NAME not set.")

    print(f"Submitting {len(ic_targets)} IC benchmark job(s)...")
    for label, ep, ic in ic_targets:
        try:
            jobs.append(submit_benchmark(label, ep, ic))
        except Exception as e:
            print(f"  FAILED '{label}': {e}")

    print(f"\n{len([j for j in jobs if 'IC' in j['label']])} IC job(s) submitted.")

---
<a id='part-b'></a>
## Part B — Non-IC Benchmark (Task 02.01)

> **Run this section if you completed task 02.01** (standard merged fine-tuned endpoint).
> Skip to [Step 6 — Poll jobs](#step-6) if you only completed 02.02.

Submits a single benchmark job against the merged SFT endpoint deployed in task 02.01.

In [ ]:
# ── Part B: Submit Non-IC benchmark job ──────────────────────────────────────
if not NON_IC_ENDPOINT:
    print("⚠️  NON_IC_ENDPOINT is not set. Did you run task 02.01?")
    print("   Set TUNED_ENDPOINT_NAME manually in Step 2 and re-run.")
else:
    print("Submitting Non-IC benchmark job...")
    try:
        jobs.append(submit_benchmark("Non-IC (merged SFT)", NON_IC_ENDPOINT, ic_name=None))
        print("\n1 Non-IC job submitted.")
    except Exception as e:
        print(f"  FAILED 'Non-IC (merged SFT)': {e}")

<a id='step-6'></a>
## Step 6 — Poll all jobs to completion

Polls every 30 seconds until every submitted job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Jobs run in parallel — **~5 min per job** in workshop mode (50 requests, concurrency 3).

> This cell covers jobs from both Part A and Part B — run it once regardless of which parts you executed.

In [ ]:
def poll_jobs(jobs, poll_interval=30):
    pending = {j["job_name"]: j for j in jobs}
    results = {}
    while pending:
        for job_name in list(pending.keys()):
            desc   = sm.describe_ai_benchmark_job(AIBenchmarkJobName=job_name)
            status = desc["AIBenchmarkJobStatus"]
            if status in ("Completed", "Failed", "Stopped"):
                label = pending[job_name]["label"]
                results[label] = {"status": status, "desc": desc, **pending.pop(job_name)}
                print(f"  [{datetime.now().strftime('%H:%M:%S')}] '{label}' -> {status}")
        if pending:
            print(f"  [{datetime.now().strftime('%H:%M:%S')}] Waiting ({len(pending)} running)...")
            time.sleep(poll_interval)
    return results

if not jobs:
    raise RuntimeError(
        "No jobs were submitted. Run Part A (IC) and/or Part B (Non-IC) above first."
    )

print(f"Polling {len(jobs)} benchmark job(s) — workshop mode ~5 min per job...")
job_results = poll_jobs(jobs)
print("\nAll jobs complete.")
for label, r in job_results.items():
    print(f"  {label}: {r['status']}")

## Step 7 — Download and parse results from S3

Each completed job writes an `output/output.tar.gz` to S3 containing:
- `profile_export_aiperf.json` — aggregated metrics (latency percentiles, throughput, TTFT, ITL)
- `profile_export.jsonl` — per-request raw data
- `plots/ttft_timeline.png` and `plots/ttft_over_time.png` — pre-generated charts

In [ ]:
def download_metrics(s3_output_location):
    parsed = s3_output_location.rstrip("/").replace("s3://", "").split("/", 1)
    bucket, prefix = parsed[0], parsed[1]

    # List objects to find the job-specific subfolder
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/")
    subfolders = [cp["Prefix"] for cp in resp.get("CommonPrefixes", [])]
    if not subfolders:
        raise FileNotFoundError(f"No output found under s3://{bucket}/{prefix}")
    job_prefix = subfolders[0]

    tar_key = f"{job_prefix}output/output.tar.gz"
    print(f"  Downloading s3://{bucket}/{tar_key}")
    tar_obj  = s3_client.get_object(Bucket=bucket, Key=tar_key)
    tar_bytes = io.BytesIO(tar_obj["Body"].read())

    extracted = {}
    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        for member in tar.getmembers():
            name = member.name.lstrip("./")
            f = tar.extractfile(member)
            if f:
                extracted[name] = f.read()
    return extracted

metrics_by_label = {}
for label, r in job_results.items():
    if r["status"] != "Completed":
        print(f"  SKIPPED '{label}' (status={r['status']})")
        continue
    try:
        s3_loc = r["desc"]["OutputConfig"]["S3OutputLocation"]
        extracted = download_metrics(s3_loc)
        json_data = extracted.get("profile_export_aiperf.json")
        if json_data:
            metrics_by_label[label] = json.loads(json_data)
            print(f"  Loaded metrics for '{label}'")
        else:
            print(f"  WARNING: profile_export_aiperf.json not found for '{label}'")
    except Exception as e:
        print(f"  ERROR loading '{label}': {e}")

print(f"\nMetrics loaded for {len(metrics_by_label)} target(s).")

## Step 8 — Summary table and comparison charts

Extracts key metrics from each job and renders a comparison table and charts. Works with any number of targets — from a single non-IC endpoint up to the full IC comparison.

In [ ]:
def extract_summary(label, m):
    return {
        "Target":                label,
        "TTFT p50 (ms)":         round(m["time_to_first_token"]["p50"],  1),
        "TTFT p90 (ms)":         round(m["time_to_first_token"]["p90"],  1),
        "TTFT p99 (ms)":         round(m["time_to_first_token"]["p99"],  1),
        "Req Latency p50 (ms)":  round(m["request_latency"]["p50"],      1),
        "Req Latency p90 (ms)":  round(m["request_latency"]["p90"],      1),
        "Req Latency p99 (ms)":  round(m["request_latency"]["p99"],      1),
        "Output Throughput (t/s)": round(m["output_token_throughput"]["avg"], 1),
        "Req Throughput (req/s)":  round(m["request_throughput"]["avg"],      2),
        "ITL p50 (ms)":          round(m["inter_token_latency"]["p50"],  2),
        "ITL p90 (ms)":          round(m["inter_token_latency"]["p90"],  2),
    }

summary_rows = [extract_summary(lbl, m) for lbl, m in metrics_by_label.items()]
summary_df   = pd.DataFrame(summary_rows).set_index("Target")

print("=== Benchmark Summary ===")
display(summary_df)

In [ ]:
# ── Chart 1: TTFT percentiles + Chart 2: Throughput ─────────────────────────
labels = list(metrics_by_label.keys())
n      = len(labels)
x      = np.arange(n)
width  = 0.25
colors = ["#4C72B0", "#DD8452", "#C44E52"]

if n == 1:
    # Single target — show as a simple bar chart instead of grouped
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    pct_vals = [
        summary_df["TTFT p50 (ms)"].values[0],
        summary_df["TTFT p90 (ms)"].values[0],
        summary_df["TTFT p99 (ms)"].values[0],
    ]
    axes[0].bar(["p50", "p90", "p99"], pct_vals, color=colors, alpha=0.85)
    axes[0].set_title(f"TTFT — {labels[0]}", fontsize=13)
    axes[0].set_ylabel("ms")
    axes[0].grid(axis="y", alpha=0.3)
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, (pct, col) in enumerate(zip(["TTFT p50 (ms)", "TTFT p90 (ms)", "TTFT p99 (ms)"], colors)):
        axes[0].bar(x + i*width, summary_df[pct].values, width, label=pct, color=col, alpha=0.85)
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels([l.replace("IC - ", "IC\n") for l in labels], fontsize=9)
    axes[0].set_title("Time to First Token (TTFT)", fontsize=13)
    axes[0].set_ylabel("ms")
    axes[0].legend(fontsize=9)
    axes[0].grid(axis="y", alpha=0.3)

# Throughput chart — same for 1 or N targets
tput = summary_df["Output Throughput (t/s)"].values
bar_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"][:n]
bars = axes[1].bar([l.replace("IC - ", "IC\n") for l in labels], tput,
                   color=bar_colors, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, tput):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{val:.0f}", ha="center", va="bottom", fontsize=10)
axes[1].set_title("Output Token Throughput (tokens/s)", fontsize=13)
axes[1].set_ylabel("tokens/s")
axes[1].grid(axis="y", alpha=0.3)

title = "SageMaker AI Benchmark — IC vs Non-IC (Qwen3-4B)" if n > 1 else f"SageMaker AI Benchmark — {labels[0]}"
plt.suptitle(title, fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("benchmark_ttft_throughput.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: benchmark_ttft_throughput.png")

In [ ]:
# ── Chart 3: Full latency percentile comparison ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
metrics_to_plot = ["Req Latency p50 (ms)", "Req Latency p90 (ms)", "Req Latency p99 (ms)"]
x     = np.arange(len(metrics_to_plot))
n     = len(labels)
width = 0.8 / max(n, 1)

for i, label in enumerate(labels):
    vals = [summary_df.loc[label, m] for m in metrics_to_plot]
    ax.bar(x + i*width - (n-1)*width/2, vals, width,
           label=label.replace("IC - ", "IC "), alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(["p50", "p90", "p99"], fontsize=11)
title = "Request Latency Percentiles — IC vs Non-IC" if n > 1 else f"Request Latency Percentiles — {labels[0]}"
ax.set_title(title, fontsize=13)
ax.set_ylabel("Latency (ms)")
if n > 1:
    ax.legend(fontsize=9, loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("benchmark_latency_pct.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: benchmark_latency_pct.png")

In [ ]:
# ── Chart 4: Inter-token latency ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
itl_p50 = summary_df["ITL p50 (ms)"].values
itl_p90 = summary_df["ITL p90 (ms)"].values
x       = np.arange(len(labels))
ax.bar(x - 0.2, itl_p50, 0.4, label="ITL p50", color="#4C72B0", alpha=0.85)
ax.bar(x + 0.2, itl_p90, 0.4, label="ITL p90", color="#DD8452", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([l.replace("IC - ", "IC\n") for l in labels], fontsize=9)
ax.set_title("Inter-Token Latency (ITL)", fontsize=13)
ax.set_ylabel("ms")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("benchmark_itl.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9 — Display pre-generated TTFT plots from S3

The Benchmarking Service generates TTFT timeline and over-time plots for each job. This cell downloads and displays them inline.

In [ ]:
def show_plots_for_job(label, r):
    if r["status"] != "Completed":
        return
    s3_loc = r["desc"]["OutputConfig"]["S3OutputLocation"]
    parsed = s3_loc.rstrip("/").replace("s3://", "").split("/", 1)
    bucket, prefix = parsed[0], parsed[1]

    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/")
    subfolders = [cp["Prefix"] for cp in resp.get("CommonPrefixes", [])]
    if not subfolders:
        return
    job_prefix = subfolders[0]

    tar_key  = f"{job_prefix}output/output.tar.gz"
    tar_obj  = s3_client.get_object(Bucket=bucket, Key=tar_key)
    tar_bytes = io.BytesIO(tar_obj["Body"].read())

    extracted = {}
    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        for member in tar.getmembers():
            name = member.name.lstrip("./")
            f = tar.extractfile(member)
            if f:
                extracted[name] = f.read()

    display(Markdown(f"### {label}"))
    for plot_name in ["plots/ttft_timeline.png", "plots/ttft_over_time.png"]:
        data = extracted.get(plot_name)
        if data:
            display(Markdown(f"**{plot_name}**"))
            display(Image(data=data, format="png"))

for label, r in job_results.items():
    show_plots_for_job(label, r)

## Key Observations

### Running modes

This notebook is split into two independent benchmark sections:

| Section | Run if you completed | What gets benchmarked |
|---|---|---|
| **Part A — IC Benchmark** | Task 02.02 | Base IC + Medical adapter IC + FinQA adapter IC |
| **Part B — Non-IC Benchmark** | Task 02.01 | Merged fine-tuned endpoint (standard deployment) |
| **Both parts** | Tasks 02.01 + 02.02 | Full IC vs Non-IC comparison |

### IC vs Non-IC trade-offs

| Dimension | Non-IC (standard endpoint) | IC-based endpoint |
|---|---|---|
| **Deployment model** | One model per endpoint | Multiple models/adapters share one endpoint |
| **Adapter switching** | Requires separate endpoint per adapter | Route to different IC at request time — zero redeploy |
| **GPU utilisation** | Dedicated GPU per endpoint | Shared GPU across ICs — better utilisation |
| **Latency overhead** | Minimal | Small IC routing overhead (~1–5 ms) |
| **Cost** | One instance per model | One instance for N adapters |
| **Benchmarking** | `BenchmarkTarget.Endpoint.Identifier` only | Add `InferenceComponents` list to target a specific IC |

### What the Benchmarking Service measures
- **TTFT** — time from request sent to first token received (streaming)
- **Request latency** — full end-to-end time including generation
- **Output token throughput** — tokens generated per second across all concurrent users
- **Inter-token latency (ITL)** — time between consecutive tokens (decode speed)

### When to use ICs
- Serving **multiple LoRA adapters** without provisioning separate instances
- **Zero-downtime adapter updates** — add/remove ICs without touching the endpoint
- **Independent scaling** of each adapter copy count

## Step 11 — Cleanup

Delete benchmark jobs and the workload config. Jobs must be in a terminal state before deletion.

In [ ]:
# Delete benchmark jobs
for j in jobs:
    try:
        sm.delete_ai_benchmark_job(AIBenchmarkJobName=j["job_name"])
        print(f"Deleted benchmark job: {j['job_name']}")
    except Exception as e:
        print(f"Could not delete {j['job_name']}: {e}")

# Delete workload config
try:
    sm.delete_ai_workload_config(AIWorkloadConfigName=WORKLOAD_NAME)
    print(f"Deleted workload config: {WORKLOAD_NAME}")
except Exception as e:
    print(f"Could not delete workload config: {e}")

### (Optional) Delete endpoints from 02.01 and 02.02

> **Warning:** Irreversible. Only run after you are done with all task_03 notebooks.

In [ ]:
# Uncomment to delete endpoints and ICs from 02.01 / 02.02
# from sagemaker.core.resources import Endpoint
#
# # Delete IC endpoint resources (from 02.02)
# for ic in [FINQA_IC_NAME, ADAPTER_IC_NAME, BASE_IC_NAME]:
#     if ic:
#         try:
#             sm.delete_inference_component(InferenceComponentName=ic)
#             print(f"Deleted IC: {ic}")
#         except Exception as e:
#             print(f"Could not delete IC {ic}: {e}")
# time.sleep(15)
#
# try:
#     Endpoint.get(endpoint_name=IC_ENDPOINT).delete()
#     print(f"Deleted IC endpoint: {IC_ENDPOINT}")
# except Exception as e:
#     print(f"Could not delete IC endpoint: {e}")
#
# # Delete non-IC endpoint (from 02.01)
# try:
#     Endpoint.get(endpoint_name=NON_IC_ENDPOINT).delete()
#     print(f"Deleted non-IC endpoint: {NON_IC_ENDPOINT}")
# except Exception as e:
#     print(f"Could not delete non-IC endpoint: {e}")